In [ ]:
import pandas as pd
import tensorflow as tf
import numpy as np
from datetime import datetime
from datetime import timedelta
from plotly_resampler import FigureResampler, register_plotly_resampler, FigureWidgetResampler
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv('pcap2ipfix-applabel-yafscii.csv')
df = df.sort_values('start-time').reset_index(drop=True)
df

In [ ]:
df['start-time'] = pd.to_datetime(df['start-time'])
df['end-time'] = pd.to_datetime(df['end-time'])
df_encoded = pd.get_dummies(df, columns=['sip', 'dip', 'sp', 'dp', 'end-reason'])
df_encoded = df_encoded.sort_values('start-time').reset_index(drop=True)
df_encoded


In [ ]:
time_step = 10

def create_dataset(data, time_step):
    x = []
    for i in range(len(data) - time_step):
        x.append(data[i:(i+time_step)])
    return np.array(x)

values = df_encoded['oct'].values
dataset = create_dataset(values, time_step)
dataset = dataset.reshape(dataset.shape[0], dataset.shape[1], 1)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, input_shape=(time_step, 1), return_sequences=True),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.RepeatVector(time_step),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

history = model.fit(dataset, dataset, epochs=1, batch_size=128, validation_split=0.3, shuffle=False)

dataset_pred = model.predict(dataset)
mse = np.mean(np.power(dataset.reshape(dataset.shape[0],time_step) - dataset_pred.reshape(dataset_pred.shape[0], time_step), 2), axis=1)

threshold = np.percentile(mse, 99)

In [ ]:
anomalies = mse > threshold
print(mse)
print(f'Threshold: {threshold}')
anomalies_index = np.where(anomalies)[0]

print(f'Number of anomalies detected: {len(anomalies_index)}')
print(f'Indices of anomalies: {anomalies_index}')
fig = go.Figure()
register_plotly_resampler(mode="auto", default_n_shown_samples=1500)

# Plot anomalies
fig.add_trace(go.Scatter(
    x=df['start-time'],
    y=df['oct'],
    mode='lines',
    name='Time Series Data'
))

fig.add_trace(go.Scatter(
    x=df['start-time'].iloc[anomalies_index],
    y=df['oct'].iloc[anomalies_index],
    mode='markers',
    marker=dict(color='red', size=5),
    name='Anomalies'
))

display(fig)